# PPP (patent–paper pair) — yearly citation trends

For every patent-paper pair in `_patent_paper_pairs_plus.csv`, build **year-by-year citation trends**:

- **Paper side** (anchored at the paper's **publication year**):
  - `p2p`  — **paper→paper** citations (how many papers cite the pair's paper each year, from the MAG graph).
  - `pat2p` — **patent→paper** citations (how many US patents cite the paper each year, by patent grant year),
    split into **examiner** / **non-examiner** (third-party excluded).
- **Patent side** (anchored at the patent's **grant year**):
  - `pat2pat` — **patent→patent** citations (how many US patents cite the pair's patent each year, by citing
    patent grant year), split into **examiner** / **non-examiner** (third-party excluded).

## Raw / input data
```
/project/jevans/Dawoon/Science of Science/PPP/_patent_paper_pairs_plus.csv   # paperid (W…), patent (US-…) pairs
/project/jevans/Dawoon/Science of Science/pcs/pcs_oa_uspto.csv               # patent->paper (PCS): reftype(app/exm), oaid, patent
/project/jevans/Dawoon/Science of Science/PatentView/Granted/g_us_patent_citation.tsv.zip  # patent->patent: patent_id, citation_patent_id, citation_category
/project/jevans/Dawoon/Science of Science/PatentView/Granted/g_patent.tsv.zip             # patent grant years
/project/jevans/Dawoon/Science of Science/OpenAlex/cache/paper_csr.npz                        # MAG citation graph (paper->paper) + paper years
```

## Output (two long tables, one row per pair × citing-year)
```
/project/jevans/Dawoon/Science of Science/PPP/output/ppp_paper_trend.parquet   # paperid, patent, pub_year, cite_year, yrs_since_pub, p2p, pat2p_examiner, pat2p_applicant
/project/jevans/Dawoon/Science of Science/PPP/output/ppp_patent_trend.parquet  # paperid, patent, grant_year, cite_year, yrs_since_grant, pat2pat_examiner, pat2pat_applicant
```
`yrs_since_pub` / `yrs_since_grant` >= 0. A pair contributes rows only for years in which it received >= 1
citation of the given type.

In [1]:
import os, sys, re, gc, time
import numpy as np, pandas as pd
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PPP')
import ppp_common as P

# Was: an os.chdir('..') hunt for Data/Reliance on Science, then Windows E: paths.
#   Data/Reliance on Science/_patent_paper_pairs_plus.csv -> P.load_pairs()  (see below)
#   Data/Reliance on Science/pcs_oa_uspto.csv             -> Science of Science/pcs/
#   E:/.../PatentView/Granted/*.tsv.zip                   -> P.granted(), local-then-project
#   E:/.../paper_disruption_csr.npz                       -> OpenAlex/cache/paper_csr.npz
#   notebook/PPP/output                                   -> Science of Science/PPP/output
ROOT    = P.BASE
PCS     = P.PCS_CSV
GPATENT = P.granted('g_patent.tsv.zip')
GCITE   = P.granted('g_us_patent_citation.tsv.zip')
CSR_NPZ = P.CSR_NPZ
OUT     = P.OUT
P.preflight()

ROOT: C:\Users\jdwoo\OneDrive\Desktop\Research\Science of Science


## 1. Load pairs → paper (MAG) & patent ids, with publication / grant years

In [2]:
%%time
# The pair list is _patent_paper_pairs_plus.csv -- the file this notebook's header has always
# named. 548,315 pairs over 335,917 papers and 309,729 patents, and it carries ppp_score,
# concepts, the paper/patent sector flags and the commercialisation flags alongside the ids.
# P.load_pairs() normalises it to paperid / patent / oaid / patent_id and passes the extras
# through, so the rest of the cell is unchanged. NB_PAIRS_SOURCE=adjusted switches back to the
# 42,967-pair finalpppsadjusted_260831.csv.
pr = P.load_pairs()

# patent grant years
gp = pd.read_csv(GPATENT, sep='\t', usecols=['patent_id', 'patent_date'], dtype={'patent_id': str})
gp['gy'] = pd.to_datetime(gp['patent_date'], errors='coerce').dt.year
pat_gy = dict(zip(gp['patent_id'], gp['gy'])); del gp; gc.collect()

# paper publication years (from CSR uni_mag/year)
z = np.load(CSR_NPZ)
uni_mag = z['uni_mag']; year = z['year'].astype(np.int32)
in_ptr, in_idx = z['in_ptr'], z['in_idx']; n = len(uni_mag)
def mag_to_code(a):
    i = np.clip(np.searchsorted(uni_mag, a), 0, n - 1); return np.where(uni_mag[i] == a, i, -1).astype(np.int64)
pr['pcode'] = mag_to_code(pr['oaid'].to_numpy())
pr['pub_year'] = np.where(pr['pcode'] >= 0, year[pr['pcode'].clip(lower=0).to_numpy()], -1)
pr['grant_year'] = pr['patent_id'].map(pat_gy)
print(f"papers matched in graph: {(pr.pcode>=0).mean()*100:.1f}% | patents with grant year: {pr.grant_year.notna().mean()*100:.1f}%")

pairs: 548,250 | distinct papers 335,896 | distinct patents 309,679


papers matched in graph: 94.1% | patents with grant year: 100.0%
CPU times: total: 22.6 s
Wall time: 36.2 s


## 2. paper→paper (p2p) yearly — from the MAG citation graph (per pair-paper)

In [3]:
%%time
pair_paper_codes = np.unique(pr.loc[pr.pcode >= 0, 'pcode'].to_numpy())
is_pair = np.zeros(n, np.bool_); is_pair[pair_paper_codes] = True
# citers of each pair-paper: iterate that paper's in-edges (citers) -> citing years
rows = []
yr = year
for code in pair_paper_codes.tolist():
    a0, a1 = in_ptr[code], in_ptr[code + 1]
    if a1 == a0: continue
    cy = yr[in_idx[a0:a1]]; cy = cy[cy >= yr[code]]         # citers from publication year onward
    if len(cy) == 0: continue
    vals, cnts = np.unique(cy, return_counts=True)
    for v, c in zip(vals.tolist(), cnts.tolist()):
        rows.append((code, v, c))
p2p = pd.DataFrame(rows, columns=['pcode', 'cite_year', 'p2p'])
print(f'p2p (paper, year) rows: {len(p2p):,}')

p2p (paper, year) rows: 2,672,640
CPU times: total: 3.66 s
Wall time: 3.7 s


## 3. patent→paper (pat2p) yearly + examiner/applicant — scan PCS (US-only)

In [4]:
%%time
paper_set = set(pr.loc[pr.pcode >= 0, 'oaid'].tolist())
pat_re = re.compile(r'us-0*([0-9]+)-')
CAT = {'exm': 'examiner'}   # examiner vs non-examiner (app + other)
acc = []; t0 = time.time(); seen = 0
for ch in pd.read_csv(PCS, usecols=['reftype', 'oaid', 'patent'], dtype=str, chunksize=5_000_000):
    oa = pd.to_numeric(ch['oaid'], errors='coerce')
    m = oa.isin(paper_set)
    if m.any():
        sub = ch[m.values]; oav = oa[m].astype(np.int64).values
        pn = sub['patent'].str.lower().str.extract(pat_re, expand=False)
        gy = pn.map(pat_gy)
        bucket = sub['reftype'].map(CAT).fillna('non_examiner').values
        d = pd.DataFrame({'oaid': oav, 'cite_year': gy.values, 'bucket': bucket})
        d = d.dropna(subset=['cite_year'])
        acc.append(d.groupby(['oaid', 'cite_year', 'bucket']).size().reset_index(name='n'))
    seen += len(ch)
pat2p = pd.concat(acc, ignore_index=True).groupby(['oaid', 'cite_year', 'bucket'])['n'].sum().reset_index()
pat2p = pat2p.pivot_table(index=['oaid', 'cite_year'], columns='bucket', values='n', fill_value=0).reset_index()
for b in ['examiner', 'non_examiner']:
    if b not in pat2p.columns: pat2p[b] = 0
pat2p = pat2p.rename(columns={'examiner': 'pat2p_examiner', 'non_examiner': 'pat2p_non_examiner'})
print(f'[{time.time()-t0:.0f}s] pat2p (paper, year) rows: {len(pat2p):,}')

[52s] pat2p (paper, year) rows: 806,982
CPU times: total: 50.9 s
Wall time: 51.7 s


## 4. patent→patent (pat2pat) yearly + examiner/applicant — scan US patent citations

In [5]:
%%time
patent_set = set(pr['patent_id'].tolist())
TP = 'third party'   # excluded; everything not 'cited by examiner' (and not third-party) is non_examiner
acc = []; t0 = time.time(); seen = 0
for ch in pd.read_csv(GCITE, sep='\t', usecols=['patent_id', 'citation_patent_id', 'citation_category'],
                      dtype=str, chunksize=5_000_000):
    ch = ch.dropna(subset=['patent_id', 'citation_patent_id'])
    cat = ch['citation_category'].fillna('').str.lower()
    m = ch['citation_patent_id'].isin(patent_set) & ~cat.str.contains(TP, na=False)  # pair-patent, third-party excluded
    if m.any():
        sub = ch[m.values]
        gy = sub['patent_id'].map(pat_gy)                  # citing patent's grant year
        bucket = np.where(cat.values[m.values] == 'cited by examiner', 'examiner', 'non_examiner')
        d = pd.DataFrame({'patent_id': sub['citation_patent_id'].values, 'cite_year': gy.values, 'bucket': bucket})
        d = d.dropna(subset=['cite_year'])
        acc.append(d.groupby(['patent_id', 'cite_year', 'bucket']).size().reset_index(name='n'))
    seen += len(ch)
pat2pat = pd.concat(acc, ignore_index=True).groupby(['patent_id', 'cite_year', 'bucket'])['n'].sum().reset_index()
pat2pat = pat2pat.pivot_table(index=['patent_id', 'cite_year'], columns='bucket', values='n', fill_value=0).reset_index()
for b in ['examiner', 'non_examiner']:
    if b not in pat2pat.columns: pat2pat[b] = 0
pat2pat = pat2pat.rename(columns={'examiner': 'pat2pat_examiner', 'non_examiner': 'pat2pat_non_examiner'})
print(f'[{time.time()-t0:.0f}s] pat2pat (patent, year) rows: {len(pat2pat):,}')

[274s] pat2pat (patent, year) rows: 1,667,434
CPU times: total: 4min 28s
Wall time: 4min 33s


## 5. Assemble per-pair trend tables and save

In [6]:
%%time
# ---- paper side: join p2p (via pcode) and pat2p (via oaid) onto pairs, anchored at pub_year ----
code2oa = dict(zip(pr.loc[pr.pcode >= 0, 'pcode'], pr.loc[pr.pcode >= 0, 'oaid']))
p2p2 = p2p.copy(); p2p2['oaid'] = p2p2['pcode'].map(code2oa)
p2p2 = p2p2[['oaid', 'cite_year', 'p2p']]
paper_year = p2p2.merge(pat2p, on=['oaid', 'cite_year'], how='outer')
pairs_paper = pr[['paperid', 'patent', 'oaid', 'pub_year']].drop_duplicates()
pt = pairs_paper.merge(paper_year, on='oaid', how='inner')
for c in ['p2p', 'pat2p_examiner', 'pat2p_non_examiner']:
    pt[c] = pt[c].fillna(0).astype('int64')
pt['cite_year'] = pt['cite_year'].astype(int)
pt = pt[pt['cite_year'] >= pt['pub_year']]
pt['yrs_since_pub'] = pt['cite_year'] - pt['pub_year']
pt = pt[['paperid', 'patent', 'pub_year', 'cite_year', 'yrs_since_pub', 'p2p', 'pat2p_examiner', 'pat2p_non_examiner']]
pt.to_parquet(os.path.join(OUT, 'ppp_paper_trend.parquet'), index=False)
print(f'ppp_paper_trend: {len(pt):,} rows')

# ---- patent side: join pat2pat onto pairs, anchored at grant_year ----
pairs_pat = pr[['paperid', 'patent', 'patent_id', 'grant_year']].dropna(subset=['grant_year']).drop_duplicates()
qt = pairs_pat.merge(pat2pat, on='patent_id', how='inner')
for c in ['pat2pat_examiner', 'pat2pat_non_examiner']:
    qt[c] = qt[c].fillna(0).astype('int64')
qt['cite_year'] = qt['cite_year'].astype(int); qt['grant_year'] = qt['grant_year'].astype(int)
qt = qt[qt['cite_year'] >= qt['grant_year']]
qt['yrs_since_grant'] = qt['cite_year'] - qt['grant_year']
qt = qt[['paperid', 'patent', 'grant_year', 'cite_year', 'yrs_since_grant', 'pat2pat_examiner', 'pat2pat_non_examiner']]
qt.to_parquet(os.path.join(OUT, 'ppp_patent_trend.parquet'), index=False)
print(f'ppp_patent_trend: {len(qt):,} rows')
display(pt.head(6)); display(qt.head(6))

ppp_paper_trend: 5,300,278 rows


ppp_patent_trend: 2,906,156 rows


,paperid,patent,pub_year,cite_year,yrs_since_pub,p2p,pat2p_examiner,pat2p_non_examiner
0,W2025049717,US-10000036,2014,2015,1,3,0,0
1,W2025049717,US-10000036,2014,2016,2,5,0,0
2,W2025049717,US-10000036,2014,2017,3,7,0,1
3,W2025049717,US-10000036,2014,2018,4,2,0,1
4,W2025049717,US-10000036,2014,2019,5,7,0,4
5,W2025049717,US-10000036,2014,2020,6,7,0,3


,paperid,patent,grant_year,cite_year,yrs_since_grant,pat2pat_examiner,pat2pat_non_examiner
0,W2025049717,US-10000036,2018,2022,4,0,1
1,W2002965373,US-10000384,2018,2025,7,0,1
2,W2059395331,US-10000410,2018,2020,2,2,0
3,W2059395331,US-10000410,2018,2022,4,1,1
4,W2059395331,US-10000410,2018,2024,6,0,1
5,W2509086927,US-10000410,2018,2020,2,2,0


CPU times: total: 4.28 s
Wall time: 4.38 s
